# Week 35, Day 3: The Poisson GLM with spike history

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w035/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A neuron with a history

Complete the simulation loop: draw the count in bin t from a Poisson distribution with mean exp(drive[t]), and if any spikes occur, add their number times h to drive[t + 1 : t + 1 + H] (cut at the end).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2005)
T, D, H = 60000, 20, 40                          # 1 ms bins, 20-bin stimulus filter, 40-bin history
x = r.normal(size=T)
lag = np.arange(D)
k = np.exp(-lag / 4) * np.sin(lag / 2.5); k = 0.6 * k / np.linalg.norm(k)
hl = np.arange(1, H + 1)
h = -5 * np.exp(-hl / 2) + 0.8 * np.exp(-((hl - 10) / 4) ** 2)
b = np.log(30 / 1000)

Xs = np.zeros((T, D))
for j in range(D):
    Xs[j:, j] = x[: T - j]
drive = Xs @ k + b
y = np.zeros(T, int)
for t in range(T):
    y[t] = ...
    if y[t]:
        ...
print(f"{y.sum()} spikes, {y.sum() / 60:.1f} Hz; shortest interval {np.diff(np.where(y > 0)[0]).min()} ms")
plt.plot(hl, h); plt.axhline(0, color="gray", lw=0.5); plt.xlabel("ms after a spike"); plt.ylabel("history filter"); plt.show()

In [ ]:
import numpy as np
def test_rate():
    "the neuron fires between 20 and 60 Hz"
    assert 20 < y.sum() / 60 < 60

def test_refractory():
    "fewer than 2% of intervals are 2 ms or shorter (a Poisson neuron at this rate would give about 7%)"
    isi = np.diff(np.where(y > 0)[0])
    assert np.mean(isi <= 2) < 0.02

test_rate()
test_refractory()
print("All checks passed")

### Exercise: A log-spaced raised-cosine basis

Complete `raised_cosines(n, H)`: centres evenly spaced on a log(t + 3) axis from log 3 to log(H + 3), each bump 0.5(1 + cos(π(log(t + 3) - c)/(2w))) within 2w of its centre and 0 outside, where w is the spacing. Then build `Yh`, the lagged spike train (T x H), and `Xh = Yh @ Bh`.

In [ ]:
def raised_cosines(n, H):
    tt = np.log(hl + 3)
    centres = np.linspace(np.log(3), np.log(H + 3), n)
    w = centres[1] - centres[0]
    cols = []
    for c in centres:
        cols.append(...)
    return np.array(cols).T

Bh = raised_cosines(6, H)
Yh = np.zeros((T, H))
for i in range(1, H + 1):
    Yh[i:, i - 1] = y[: T - i]
Xh = ...
print("basis shape", Bh.shape, "; design matrix", Xh.shape)
plt.plot(hl, Bh); plt.xlabel("ms after a spike"); plt.show()

In [ ]:
import numpy as np
def test_basis():
    "Bh has 40 rows and 6 columns, each a non-negative bump"
    assert Bh.shape == (40, 6) and (Bh >= 0).all() and (Bh.max(0) > 0.8).all()

def test_design():
    "Xh is the past spike train projected onto the basis"
    assert np.allclose(Xh, Yh @ Bh)

test_basis()
test_design()
print("All checks passed")

### Exercise: Recovering both filters

Write `fit_glm(X, y)`, Newton's method for the Poisson GLM with an added constant column, starting from w = 0 except the bias at log(mean y). Fit the LNP model (stimulus only) and the GLM (stimulus plus history basis) on the first 45 s, and reconstruct the fitted history filter `h_fit = Bh @ w_glm[D:D+6]`.

In [ ]:
def fit_glm(X, y, iters=25):
    X = np.c_[X, np.ones(len(y))]
    w = np.zeros(X.shape[1]); w[-1] = np.log(y.mean())
    for _ in range(iters):
        lam = np.exp(X @ w)
        grad = ...
        hess = ...
        w = w + np.linalg.solve(hess + 1e-6 * np.eye(len(w)), grad)
    return w

train = np.arange(T) < 45000
w_lnp = fit_glm(Xs[train], y[train])
w_glm = fit_glm(np.c_[Xs, Xh][train], y[train])
h_fit = Bh @ w_glm[D:D + 6]
print("history filter, first 12 ms: fitted", np.round(h_fit[:12], 2))
print("                             true  ", np.round(h[:12], 2))
plt.plot(hl, h, label="true"); plt.plot(hl, h_fit, label="fitted"); plt.legend(); plt.xlabel("ms"); plt.show()

In [ ]:
import numpy as np
def test_stim():
    "both models recover the stimulus filter direction (similarity above 0.95)"
    c = lambda v: v @ k / (np.linalg.norm(v) * np.linalg.norm(k))
    assert c(w_lnp[:D]) > 0.95 and c(w_glm[:D]) > 0.95

def test_hist():
    "the GLM's fitted history filter is strongly negative in the first 2 ms and positive near 10 ms"
    assert h_fit[:2].mean() < -2 and h_fit[8:12].mean() > 0

test_stim()
test_hist()
print("All checks passed")